# Sutton & Barto Chapter 2: 2.4-2.5
## Incremental Implementation and Nonstationary Bandits

This notebook builds a minimal implementation of Sutton and Barto's **10-armed testbed** for two ideas from Chapter 2:

- **2.4 Incremental Implementation**: update action values without storing the full reward history.
- **2.5 Tracking a Nonstationary Problem**: use a constant step size to adapt when the true action values drift.

We keep the code intentionally small so the core ideas stay visible.

## What You Will See

- a minimal stationary 10-armed testbed
- the incremental sample-average update from Section 2.4
- a nonstationary random-walk bandit from Section 2.5
- a comparison between sample averages and a constant step size

Dependencies: `numpy`, `matplotlib`

## Theory

For action-value estimation, the sample-average update can be written incrementally as

$$
Q_{n+1} = Q_n + \frac{1}{n}(R_n - Q_n).
$$

This is equivalent to averaging all observed rewards, but it only needs the current estimate and the visit count.

For **nonstationary** problems, sample averages react too slowly because old rewards never really disappear. A common fix is a constant step size:

$$
Q_{t+1} = Q_t + \alpha (R_t - Q_t),
$$

where recent rewards get more weight than old ones. That makes the estimate track drifting action values better.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

In [ ]:
class TenArmedBandit:
    def __init__(self, k=10, nonstationary=False, drift_std=0.01, seed=None):
        self.k = k
        self.nonstationary = nonstationary
        self.drift_std = drift_std
        self.rng = np.random.default_rng(seed)
        self.q_true = self.rng.normal(0.0, 1.0, size=k)

    def step(self, action):
        reward = self.rng.normal(self.q_true[action], 1.0)
        optimal = int(action == np.argmax(self.q_true))
        if self.nonstationary:
            self.q_true += self.rng.normal(0.0, self.drift_std, size=self.k)
        return reward, optimal


In [ ]:
def run_epsilon_greedy(steps=1000, epsilon=0.1, alpha=None, nonstationary=False, seed=None):
    bandit = TenArmedBandit(nonstationary=nonstationary, seed=seed)
    rng = np.random.default_rng(seed)
    q_est = np.zeros(bandit.k)
    counts = np.zeros(bandit.k, dtype=int)
    rewards = np.zeros(steps)
    optimal_actions = np.zeros(steps)

    for t in range(steps):
        if rng.random() < epsilon:
            action = rng.integers(bandit.k)
        else:
            action = rng.choice(np.flatnonzero(q_est == q_est.max()))

        reward, optimal = bandit.step(action)
        counts[action] += 1
        step_size = alpha if alpha is not None else 1.0 / counts[action]
        q_est[action] += step_size * (reward - q_est[action])

        rewards[t] = reward
        optimal_actions[t] = optimal

    return rewards, optimal_actions


def average_runs(runs=200, **kwargs):
    reward_history = []
    optimal_history = []
    for seed in range(runs):
        rewards, optimal = run_epsilon_greedy(seed=seed, **kwargs)
        reward_history.append(rewards)
        optimal_history.append(optimal)
    return np.mean(reward_history, axis=0), 100 * np.mean(optimal_history, axis=0)

## 2.4 Incremental sample-average method

Here we run a standard stationary 10-armed testbed with an $\epsilon$-greedy agent. The important point is that the estimate update uses only:

- the previous estimate `q_est[action]`
- the count `N(a)`
- the latest reward

So the algorithm is memory-efficient and matches the book's incremental derivation.

In [ ]:
avg_reward_stationary, opt_stationary = average_runs(
    runs=200,
    steps=1000,
    epsilon=0.1,
    alpha=None,
    nonstationary=False,
)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(avg_reward_stationary)
ax[0].set_title('Stationary Testbed: Average Reward')
ax[0].set_xlabel('Steps')
ax[0].set_ylabel('Average reward')

ax[1].plot(opt_stationary)
ax[1].set_title('Stationary Testbed: % Optimal Action')
ax[1].set_xlabel('Steps')
ax[1].set_ylabel('% optimal action')
plt.tight_layout()

## 2.5 Tracking a nonstationary problem

Now the true action values drift after every step by a small Gaussian random walk. We compare:

- **Sample average**: `alpha=None`
- **Constant step size**: `alpha=0.1`

The second method should adapt better because it emphasizes recent rewards.

In [ ]:
avg_reward_sa, opt_sa = average_runs(
    runs=200,
    steps=2000,
    epsilon=0.1,
    alpha=None,
    nonstationary=True,
)

avg_reward_cs, opt_cs = average_runs(
    runs=200,
    steps=2000,
    epsilon=0.1,
    alpha=0.1,
    nonstationary=True,
)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(avg_reward_sa, label='sample average')
ax[0].plot(avg_reward_cs, label='constant step size (alpha=0.1)')
ax[0].set_title('Nonstationary Testbed: Average Reward')
ax[0].set_xlabel('Steps')
ax[0].set_ylabel('Average reward')
ax[0].legend()

ax[1].plot(opt_sa, label='sample average')
ax[1].plot(opt_cs, label='constant step size (alpha=0.1)')
ax[1].set_title('Nonstationary Testbed: % Optimal Action')
ax[1].set_xlabel('Steps')
ax[1].set_ylabel('% optimal action')
ax[1].legend()
plt.tight_layout()

## Takeaway

- Incremental sample averages give the same estimate as the arithmetic mean without storing reward histories.
- In a stationary problem, sample averages are natural and stable.
- In a nonstationary problem, a constant step size usually performs better because it keeps tracking recent changes.

## Reference

Richard S. Sutton and Andrew G. Barto, *Reinforcement Learning: An Introduction*, Chapter 2.